In [34]:
from scipy.interpolate import interp1d
from scipy.integrate import solve_ivp
from scipy.optimize import root
import matplotlib.pyplot as plt

In [35]:
# Physical constants of the TOV Equations and Energy Conversions
 
import math
import numpy as np


# Fundamental constants in SI
G = 6.67430e-11            # gravitational constant [m^3 kg^-1 s^-2]
c = 2.99792458e8           # speed of light [m s^-1]
hbar = 1.054571817e-34     # reduced Planck constant [J s]
M_sun = 1.98847e30         # solar mass [kg]


# Basic unit conversions

eV_to_J = 1.602176634e-19
MeV_to_J = 1.0e6 * eV_to_J

J_to_eV = 1.0 / eV_to_J
J_to_MeV = 1.0 / MeV_to_J

fm_to_m = 1.0e-15
m_to_fm = 1.0 / fm_to_m

km_to_m = 1.0e3
m_to_km = 1.0 / km_to_m



# Energy conversions


def MeV_to_joule(E_MeV):
    """Convert energy from MeV to joules."""
    return E_MeV * MeV_to_J


def joule_to_MeV(E_J):
    """Convert energy from joules to MeV."""
    return E_J * J_to_MeV


# ============================================================
# Mass conversions
# ============================================================

def kg_to_MeV_c2(m_kg):
    """
    Convert mass in kg to equivalent rest energy in MeV/c^2.
    Uses E = m c^2.
    """
    return m_kg * c**2 * J_to_MeV


def MeV_c2_to_kg(m_MeV):
    """
    Convert mass in MeV/c^2 to kg.
    """
    return m_MeV * MeV_to_J / c**2


# ============================================================
# Energy-density / pressure conversions
# ============================================================

# 1 MeV/fm^3 in J/m^3
MeV_fm3_to_J_m3 = MeV_to_J / fm_to_m**3

# inverse conversion
J_m3_to_MeV_fm3 = 1.0 / MeV_fm3_to_J_m3


def MeV_fm3_to_SI(x):
    """
    Convert pressure or energy density:
    MeV/fm^3 -> J/m^3 = Pa
    """
    return x * MeV_fm3_to_J_m3


def SI_to_MeV_fm3(x):
    """
    Convert pressure or energy density:
    J/m^3 = Pa -> MeV/fm^3
    """
    return x * J_m3_to_MeV_fm3


# ============================================================
# Mass-density / energy-density conversions
# ============================================================

def rho_to_epsilon(rho):
    """
    Mass density [kg/m^3] -> energy density [J/m^3]
    """
    return rho * c**2


def epsilon_to_rho(epsilon):
    """
    Energy density [J/m^3] -> mass density [kg/m^3]
    """
    return epsilon / c**2


def rho_to_MeV_fm3(rho):
    """
    Mass density [kg/m^3] -> equivalent energy density [MeV/fm^3]
    """
    epsilon = rho * c**2
    return SI_to_MeV_fm3(epsilon)

In [36]:
# Equation of state: Polytropic eos from reference

K = 0.02
Gamma = 2.5
def toy_eos(rho):
    return K * np.maximum(rho,0.0) ** gamma

def densityEOS(P):

    P = np.maximum(P, 0.0)
    return (P / K)**(1.0 / Gamma)

def int_energy(P):
    P = np.maximum(P, 0.0)
    return P/(gamma - 1)

def e_density(P):
   

    rho = densityEOS(P)

    epsilon_int = int_energy(P)

    return rho * c**2 + epsilon_int
# we have values for all necessary variants of our toy eos

In [37]:
# TOV Equations
def tov_equations(r,y):
    m,P = y
    if r <= 0.0:

        raise ValueError("TOV equations cannot be evaluated at singularity.")

    if P <= 0.0:

        return [0.0, 0.0]
    E = e_density(P)
    
    dmdr = 4.0 * np.pi * r**2 * E/ c**2
    t1 = G * E * m / (c**2 * r**2)

    t2 = 1.0 + P / E

    t3 = 1.0 + (4.0 * np.pi * r**3 * P) / (m * c**2)

    t4 = 1.0 - (2.0 * G * m) / (c**2 * r)

    dPdr = -t1 * t2 * t3 / t4
    return [dPdr, dmdr]

In [38]:
#Boundary Conditions: Sueface of the Neutron Star

# Stellar surface: P(R) = 0

r_max = 5.0e4
def surface_event(r, y):
    m, P = y

    return P

surface_event.terminal = True


surface_event.direction = -1
rho_c = 1.0e18

# ============================================================
# Solve one neutron star
# ============================================================

def solve_star(rho_c,r_start=1.0e-3,r_max=5.0e4,rtol=1.0e-7,atol = [1.0e-6, 1.0e18],max_step=50.0):
    r_centre = 0.0
    m_centre = 0.0
    rho_centre = rho_c
    P_c = toy_eos(rho_c)


    # Central total energy density

    E_c = e_density(P_c)


    # ========================================================
    # 2. NUMERICAL STARTING POINT
    # ========================================================
    #
    # The physical centre is r = 0.
    #
    # But we start numerical integration at a tiny
    # positive radius r_start.
    # ========================================================


    # Small-r expansion of the enclosed mass:
    #
    # m(r) ≈ (4π/3) r^3 E_c/c^2

    m_start = ((4.0 * np.pi / 3.0)* r_start **3 * E_c / c**2)


    # To leading order near the centre:
    #
    # P(r_start) ≈ P_c

    P_start = P_c


    # Numerical initial state

    y_start = [m_start,P_start]


    # ========================================================
    # 3. INTEGRATE OUTWARD
    # ========================================================

    solution = solve_ivp(tov_equations,t_span=(r_start,r_max),y0=y_start,events=surface_event,rtol=1.0e-7,atol = [1.0e-6, 1.0e18],max_step=50)


    # ========================================================
    # 4. CHECK THAT THE SURFACE WAS REACHED
    # ========================================================

    if len(solution.t_events[0]) == 0:

        raise RuntimeError("The integration did not reach P = 0. ""Increase r_max or check the EOS.")


    # ========================================================
    # 5. EXTRACT SURFACE VALUES
    # ========================================================

    R = solution.t_events[0][0]

    M = solution.y_events[0][0][0]


    # By definition at the surface

    P_surface = 0.0

    rho_surface = densityEOS(P_surface)

    E_surface = e_density(P_surface)


    return (solution,R,M,P_c,E_c,rho_surface,E_surface)

solution, R, M, P_c, E_c, rho_surface, E_surface = solve_star(rho_c)
print(R)
print(M)
print(P_c)
print(E_c)
print(rho_surface)
print(E_surface)

RuntimeError: The integration did not reach P = 0. Increase r_max or check the EOS.